In [1]:
import sys
sys.path.append('../src')
import numpy as np
import pandas as pd

df = pd.read_csv("../data/processed/user_segments.csv")

print(df['churned_next_30d'].value_counts(normalize=True))

churned_next_30d
0    0.9586
1    0.0414
Name: proportion, dtype: float64


In [2]:
import sys
sys.path.append('../src')
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, classification_report, roc_curve
import matplotlib.pyplot as plt

df = pd.read_csv("../data/processed/user_segments.csv")

churn_features = [
    'obs_avg_sessions_per_week', 'avg_completion_rate', 'genre_diversity',
    'tenure_days', 'obs_recent_30d_sessions', 'obs_prior_30d_sessions', 'obs_session_trend_ratio',
]

X = df[churn_features]
y = df['churned_next_30d']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train churn rate:", y_train.mean())
print("Test churn rate:", y_test.mean())
print("Train size:", len(X_train), "| Test size:", len(X_test))

Train churn rate: 0.0415
Test churn rate: 0.041
Train size: 4000 | Test size: 1000


In [3]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight='balanced',
    random_state=42,
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_pred_proba = rf.predict_proba(X_test)[:, 1]  # probability of churn, class 1

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

              precision    recall  f1-score   support

           0       1.00      0.85      0.92       959
           1       0.22      1.00      0.37        41

    accuracy                           0.86      1000
   macro avg       0.61      0.93      0.64      1000
weighted avg       0.97      0.86      0.90      1000

ROC-AUC: 0.9282662326101885
